<a href="https://colab.research.google.com/github/twyeh/university-physics/blob/main/rocket_simulator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# ---------------------------------------------------------------------
# Rocket Landing Simulator (Pygame Version)
# Gate Check: 4
# ---------------------------------------------------------------------
import pygame, random, math, os
import rocket_ai_solution_pygame as rocket_ai
import rocket_ga_pygame as rocket_ga

# 確保工作目錄切換到程式所在資料夾 (相容於 Jupyter Notebook/Colab)
try:
    file_path = os.path.dirname(os.path.abspath(__file__))
    os.chdir(file_path)
except NameError:
    # 在 Colab 或 Jupyter 中 __file__ 未定義時，預設使用目前工作目錄
    os.chdir(os.getcwd())

# 初始化 Pygame
pygame.init()

# CONSTANTS
WINDOW_WIDTH  = 1800
WINDOW_HEIGHT = 800
BOAT_WIDTH = 120
BOAT_HEIGHT = 30
ROCKET_WIDTH = 50
ROCKET_HEIGHT = ROCKET_WIDTH
GRAVITY = 0.08

# 顏色定義
BLACK = (0, 0, 0)
WHITE = (255, 255, 255)
GREEN = (0, 255, 0)
LIGHT_BLUE = (173, 216, 230)
YELLOW = (255, 255, 0)
GRAY = (150, 150, 150)
RED = (255, 80, 80)

# 建立視窗、時脈與字型
screen = pygame.display.set_mode((WINDOW_WIDTH, WINDOW_HEIGHT))
pygame.display.set_caption(f"Rocket Landing Simulator - {rocket_ai.get_student_name()}")
clock = pygame.time.Clock()
hud_font = pygame.font.SysFont("arial", 20)

# 載入圖片資源 (含防呆機制)
try:
    boat_img = pygame.image.load("boat.png").convert_alpha()
    boat_img = pygame.transform.scale(boat_img, (BOAT_WIDTH, BOAT_HEIGHT))
except FileNotFoundError:
    boat_img = None

try:
    rocket_img = pygame.image.load("rocket.png").convert_alpha()
    rocket_img = pygame.transform.scale(rocket_img, (ROCKET_WIDTH, ROCKET_HEIGHT))
except FileNotFoundError:
    rocket_img = None

# Performance Variables
time_elapsed = 0
fuel_consumed = 0
num_crashes = 0
num_landings = 0
max_score = -999999

# Simulation Variables
NUM_RUNS_PER_SCENARIO = 10
num_runs = 0
fast_mode = False  # 按 F 鍵可切換高速訓練模式

# Terrain
terrain_list = []
GROUND_HEIGHT = 0
WATER_HEIGHT  = 0
GROUND_LENGTH = 0

# Rocket
rocket_x = 0
rocket_y = 0
rocket_vx = 0
rocket_vy = 0
rocket_boosting = True
rocket_thrust_up = 0
rocket_thrust_right = 0
rocket_thrust_left = 0

# Boat (i.e., Landing Pad)
boat_start_x = 0
boat_start_y = 0
boat_start_vx = 0
boat_x = 0
boat_y = 0
boat_vx = 0

# --------------------------------------------------------------
# Initializes the Simulation
# --------------------------------------------------------------
def initialize_simulation(generate_new_scenario):
    global time_elapsed, fuel_consumed
    initialize_terrain(generate_new_scenario)
    initialize_boat(generate_new_scenario)
    initialize_rocket(generate_new_scenario)
    time_elapsed = 0
    fuel_consumed = 0

# --------------------------------------------------------------
# Initializes the Terrain
# --------------------------------------------------------------
def initialize_terrain(generate_new_scenario):
    global terrain_list, GROUND_LENGTH, GROUND_HEIGHT, WATER_HEIGHT

    if generate_new_scenario == True:
        terrain_list = []

        GROUND_LENGTH = random.randint(int(0.1 * WINDOW_WIDTH), int(0.2 * WINDOW_WIDTH))
        GROUND_HEIGHT = random.randint(100, int(0.4 * WINDOW_HEIGHT))
        WATER_HEIGHT  = random.randint(50, GROUND_HEIGHT)

        for x in range(0, WINDOW_WIDTH):
            if x > GROUND_LENGTH:
                terrain_list.append(WATER_HEIGHT)
            else:
                terrain_list.append(GROUND_HEIGHT)

# --------------------------------------------------------------
# Initializes the Boat
# --------------------------------------------------------------
def initialize_boat(generate_new_scenario):
    global boat_start_x, boat_start_y, boat_start_vx, boat_x, boat_y, boat_vx

    if generate_new_scenario == True:
        boat_start_x = random.randint(int(WINDOW_WIDTH * 0.5), int(WINDOW_WIDTH - BOAT_WIDTH))
        boat_start_y = WINDOW_HEIGHT - WATER_HEIGHT
        boat_start_vx = random.random() * 2.0 - 1.0

    boat_x = boat_start_x
    boat_y = boat_start_y
    boat_vx = boat_start_vx

# --------------------------------------------------------------
# Initializes the Rocket
# --------------------------------------------------------------
def initialize_rocket(generate_new_scenario):
    global rocket_x, rocket_y, rocket_vx, rocket_vy, rocket_boosting
    rocket_x = GROUND_LENGTH * 0.5
    rocket_y = WINDOW_HEIGHT - GROUND_HEIGHT
    rocket_vx = 0
    rocket_vy = 0
    rocket_boosting = True

# --------------------------------------------------------------
# Clears the screen
# --------------------------------------------------------------
def erase_objects():
    screen.fill(BLACK)

# --------------------------------------------------------------
# Draws all of the in game objects
# --------------------------------------------------------------
def draw_objects():
    draw_terrain()
    draw_boat()
    draw_rocket()
    draw_hud()

# --------------------------------------------------------------
# Draws the Terrain
# --------------------------------------------------------------
def draw_terrain():
    for x in range(0, WINDOW_WIDTH):
        if x > GROUND_LENGTH:
            pygame.draw.line(screen, LIGHT_BLUE, (x, WINDOW_HEIGHT), (x, WINDOW_HEIGHT - terrain_list[x]))
        else:
            pygame.draw.line(screen, GREEN, (x, WINDOW_HEIGHT), (x, WINDOW_HEIGHT - terrain_list[x]))

# --------------------------------------------------------------
# Draws the Boat
# --------------------------------------------------------------
def draw_boat():
    rect_pos = (boat_x, boat_y - 25)
    if boat_img:
        screen.blit(boat_img, rect_pos)
    else:
        pygame.draw.rect(screen, GRAY, (*rect_pos, BOAT_WIDTH, BOAT_HEIGHT))

# --------------------------------------------------------------
# Draws the Rocket (and Thrusters)
# --------------------------------------------------------------
def draw_rocket():
    rect_pos = (rocket_x - ROCKET_WIDTH / 2, rocket_y - ROCKET_HEIGHT)
    if rocket_img:
        screen.blit(rocket_img, rect_pos)
    else:
        pygame.draw.rect(screen, RED, (*rect_pos, ROCKET_WIDTH, ROCKET_HEIGHT))

    if rocket_thrust_up > 0:
        pygame.draw.circle(screen, YELLOW, (int(rocket_x), int(rocket_y)), 4)
    if rocket_thrust_right > 0:
        pygame.draw.circle(screen, YELLOW, (int(rocket_x - ROCKET_WIDTH / 3), int(rocket_y - ROCKET_HEIGHT / 5)), 4)
    if rocket_thrust_left > 0:
        pygame.draw.circle(screen, YELLOW, (int(rocket_x + ROCKET_WIDTH / 3), int(rocket_y - ROCKET_HEIGHT / 5)), 4)
    if rocket_y < 0:
        pygame.draw.line(screen, YELLOW, (int(rocket_x - ROCKET_WIDTH / 2), 5), (int(rocket_x + ROCKET_WIDTH / 2), 5), 5)

# --------------------------------------------------------------
# Draws the On Screen Text
# --------------------------------------------------------------
def draw_hud():
    display_max = round(max_score, 1) if max_score > -999999 else 0
    speed_mode = "FAST (Press F for Normal)" if fast_mode else "NORMAL (Press F for Fast)"

    hud_lines = [
        f"Max Score: {display_max}  |  AI Run: {num_runs + 1}/{NUM_RUNS_PER_SCENARIO}",
        f"Time Elapsed: {time_elapsed}",
        f"Fuel Consumed: {round(fuel_consumed, 2)}",
        f"X Velocity: {round(rocket_vx, 2)}",
        f"Y Velocity: {round(rocket_vy, 2)}",
        f"Crashes: {num_crashes}  Landings: {num_landings}  |  Speed: {speed_mode}"
    ]
    for i, text in enumerate(hud_lines):
        text_surface = hud_font.render(text, True, WHITE)
        screen.blit(text_surface, (5, i * 25))

# --------------------------------------------------------------
# Updates all animated objects
# --------------------------------------------------------------
def update_objects():
    update_boat()
    update_rocket()

# --------------------------------------------------------------
# Updates the Rocket
# --------------------------------------------------------------
def update_rocket():
    global rocket_x, rocket_y, rocket_vy, rocket_vx, rocket_boosting, rocket_thrust_up, rocket_thrust_right, rocket_thrust_left, fuel_consumed

    # Launch
    if rocket_boosting == True:
        rocket_thrust_up = 0
        rocket_thrust_right = 0
        rocket_thrust_left = 0

        if rocket_y < WINDOW_HEIGHT / 2:
            rocket_thrust_right = 0.25
        else:
            rocket_thrust_up = 0.35

        if rocket_x > GROUND_LENGTH:
            rocket_boosting = False

    # Updating Velocity
    rocket_vy -= rocket_thrust_up
    rocket_vx -= abs(rocket_thrust_left)
    rocket_vx += abs(rocket_thrust_right)

    # Updating Fuel Consumed
    fuel_consumed = fuel_consumed + abs(rocket_thrust_up) + abs(rocket_thrust_right) + abs(rocket_thrust_left)

    # Updating Position
    rocket_x += rocket_vx
    rocket_y += rocket_vy
    rocket_vy += GRAVITY

# --------------------------------------------------------------
# Updates the Landing Pad / Boat
# --------------------------------------------------------------
def update_boat():
    global boat_x, boat_vx
    boat_x += boat_vx

    if boat_x < WINDOW_WIDTH / 2 or boat_x + BOAT_WIDTH > WINDOW_WIDTH:
        boat_vx *= -1

# --------------------------------------------------------------
# Checks for Manual or AI Input
# --------------------------------------------------------------
def get_input():
    global rocket_thrust_up, rocket_thrust_right, rocket_thrust_left

    rocket_thrust_up = 0
    rocket_thrust_right = 0
    rocket_thrust_left = 0

    if rocket_boosting == False:
        ai_decision = rocket_ai.run_autopilot(num_runs, rocket_x - ROCKET_WIDTH / 2, rocket_y, rocket_vx, rocket_vy, ROCKET_WIDTH, boat_x, boat_y, BOAT_WIDTH)
        rocket_thrust_left = ai_decision[0]
        rocket_thrust_right = ai_decision[1]
        rocket_thrust_up = ai_decision[2]

        keys = pygame.key.get_pressed()
        if keys[pygame.K_LEFT]:
            rocket_thrust_left = 0.1
        if keys[pygame.K_RIGHT]:
            rocket_thrust_right = 0.1
        if keys[pygame.K_UP]:
            rocket_thrust_up = 0.5

# --------------------------------------------------------------
# Detects if the Rocket has hit the ground or a boundary
# --------------------------------------------------------------
def is_simulation_over():
    if rocket_boosting == False:
        rocket_left = int(rocket_x - ROCKET_WIDTH/2)
        rocket_right = int(rocket_x + ROCKET_WIDTH/2)

        # Checks for Collisions with the Sides
        if rocket_left < 0 or rocket_right >= WINDOW_WIDTH:
            return True

        # Checks for Collisions with the Ground
        for x in range(rocket_left, rocket_right):
            if (rocket_y >= WINDOW_HEIGHT - terrain_list[x]):
                return True

    return False

# --------------------------------------------------------------
# Analyzes the Results of the Simulation
# --------------------------------------------------------------
def analyze_results():
    global max_score, num_landings, num_crashes

    rocket_left = int(rocket_x - ROCKET_WIDTH/2)
    rocket_right = int(rocket_x + ROCKET_WIDTH/2)

    w1 = 0.4
    w2 = 0.4
    w3 = 0.1
    w4 = 0.1

    if rocket_left >= boat_x and rocket_right <= boat_x + BOAT_WIDTH:
        score = w1 * (100 / rocket_vy) + w2 * (abs(100 / rocket_vx)) + (-w3 * fuel_consumed) + (-w4 * time_elapsed)
        num_landings += 1
    else:
        score = -2000
        num_crashes += 1

    rocket_ga.score_ai(num_runs, score)
    max_score = max(score, max_score)

# -----------------------------------------------------
# "Main Program"
# -----------------------------------------------------
# 注意：若 pygame 未實作 open_window，此處使用 pygame 內建顯示功能載入
# 如果是自定義庫，可保持以下邏輯或調用標準 pygame.display.set_mode
pygame.display.set_caption("Rocket Landing Simulator")

# Initializes the Simulation At Least Once
initialize_simulation(True)

# 遊戲主迴圈標記
running = True

# Main "Game Loop"
while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    if is_simulation_over() == False:
        erase_objects()
        draw_objects()
        get_input()
        update_objects()
        time_elapsed = time_elapsed + 1
    else:
        analyze_results()
        num_runs = num_runs + 1
        if num_runs == NUM_RUNS_PER_SCENARIO:
            num_runs = 0
            initialize_simulation(True)
        else:
            initialize_simulation(False)

    pygame.display.flip()
    clock.tick(60)

pygame.quit()


  AI #0 (Alive: 1 Gen):  Score: 0.00, Horiz Thrust: 0.41 (7.61 MAX), Vert Thust: 5.59 (13.93 MAX)
  Updating Score for AI 0 (-22.61): 0.00 --> -22.61
  AI #1 (Alive: 1 Gen):  Score: 0.00, Horiz Thrust: 1.51 (9.07 MAX), Vert Thust: 3.21 (11.23 MAX)
  Updating Score for AI 1 (-36.99): 0.00 --> -36.99
  AI #2 (Alive: 1 Gen):  Score: 0.00, Horiz Thrust: 0.03 (4.28 MAX), Vert Thust: 1.81 (8.46 MAX)
  Updating Score for AI 2 (-22.15): 0.00 --> -22.15
  AI #3 (Alive: 1 Gen):  Score: 0.00, Horiz Thrust: 2.89 (6.35 MAX), Vert Thust: 1.32 (11.52 MAX)
  Updating Score for AI 3 (43.51): 0.00 --> 43.51
  AI #4 (Alive: 1 Gen):  Score: 0.00, Horiz Thrust: 0.84 (2.98 MAX), Vert Thust: 2.55 (5.12 MAX)
  Updating Score for AI 4 (-2000.00): 0.00 --> -2000.00
  AI #5 (Alive: 1 Gen):  Score: 0.00, Horiz Thrust: 0.31 (1.27 MAX), Vert Thust: 1.78 (10.53 MAX)
  Updating Score for AI 5 (-2000.00): 0.00 --> -2000.00
  AI #6 (Alive: 1 Gen):  Score: 0.00, Horiz Thrust: 2.57 (5.61 MAX), Vert Thust: 0.12 (3.08 MAX)

KeyboardInterrupt: 